# 02 — Nordigen / GoCardless Bank Account Data PoC

**Decision gate (Faza 2 planu).** Sprawdzamy czy GoCardless BAD (dawniej Nordigen) wspiera polskie banki, których używamy (Pekao, Revolut, ewentualnie PKO/mBank/ING).

## Setup
1. Konto na <https://bankaccountdata.gocardless.com/> (free tier — sandbox).
2. W panelu *User secrets* utwórz parę `SECRET_ID` / `SECRET_KEY`.
3. W terminalu (PowerShell):
    ```powershell
    $env:NORDIGEN_SECRET_ID = '...'
    $env:NORDIGEN_SECRET_KEY = '...'
    ```
4. Restart Jupyter / VSCode kernel żeby zmienne były widoczne.

Notebook nie commituje sekretów — czyta je tylko ze środowiska.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

HAS_CREDS = bool(os.environ.get('NORDIGEN_SECRET_ID') and os.environ.get('NORDIGEN_SECRET_KEY'))
print('Credentials in env:', HAS_CREDS)

## 1. Autoryzacja

In [ ]:
from finance.ingestion.nordigen import NordigenClient

client = None
if HAS_CREDS:
    client = NordigenClient.from_env()
    token = client.authenticate()
    print('OK, access token len:', len(token))
else:
    print('Brak sekretów — kolejne komórki z zapytaniami HTTP zostaną pominięte.')

## 2. Lista instytucji w PL

In [ ]:
import pandas as pd

TARGET_BANKS = ['pekao', 'revolut', 'pko', 'mbank', 'ing', 'santander', 'millennium']

if client is not None:
    pl = client.list_institutions('PL')
    inst = pd.DataFrame(pl)[['id', 'name', 'transaction_total_days', 'countries']]
    print(f'Liczba instytucji PL: {len(inst)}')
    mask = inst['name'].str.lower().apply(lambda n: any(b in n for b in TARGET_BANKS))
    display(inst[mask].sort_values('name'))
else:
    inst = pd.DataFrame()
    print('skip')

## 3. Sandbox requisition (test flow)

GoCardless udostępnia instytucję **`SANDBOXFINANCE_SFIN0000`** — fikcyjny bank, na którym można przetestować pełny flow OAuth bez prawdziwego konta.

Krok stworzy tymczasowy URL — w przeglądarce zaakceptujesz "zgodę" i wrócisz na `redirect`. Potem `get_requisition(...)` zwróci listę `accounts`.

In [ ]:
import uuid

SANDBOX_ID = 'SANDBOXFINANCE_SFIN0000'
REDIRECT_URL = 'http://localhost:8000/nordigen/callback'

if client is not None:
    req = client.create_requisition(
        institution_id=SANDBOX_ID,
        redirect_url=REDIRECT_URL,
        reference=f'poc-{uuid.uuid4().hex[:8]}',
    )
    print('Requisition id:', req['id'])
    print('Otworz w przegladarce:')
    print(req['link'])
else:
    req = None
    print('skip')

Po zaakceptowaniu zgody w przeglądarce uruchom następną komórkę. Powinna pokazać listę `accounts` (id-y kont z piaskownicy).

In [ ]:
if client is not None and req is not None:
    state = client.get_requisition(req['id'])
    print('Status:', state.get('status'))
    print('Accounts:', state.get('accounts'))
    accounts = state.get('accounts') or []
else:
    accounts = []
    print('skip')

## 4. Pobranie transakcji z konta sandbox

In [ ]:
if accounts and client is not None:
    acc_id = accounts[0]
    txs = client.get_account_transactions(acc_id, date_from='2025-01-01', date_to='2026-04-30')
    booked = txs.get('transactions', {}).get('booked', [])
    print(f'Booked transactions: {len(booked)}')
    if booked:
        df = pd.json_normalize(booked)
        cols = [c for c in [
            'bookingDate', 'valueDate', 'transactionAmount.amount', 'transactionAmount.currency',
            'creditorName', 'debtorName', 'remittanceInformationUnstructured',
        ] if c in df.columns]
        display(df[cols].head(20))
else:
    print('skip (brak accounts)')

## 5. Wnioski (uzupełnij po uruchomieniu)

Po wykonaniu komórek wypełnij decyzję w `docs/adr/0001-nordigen.md`:

- Czy **Pekao SA** jest na liście instytucji PL?
- Czy **Revolut** jest pokryty (zwykle przez `REVOLUT_REVOLT21` / `REVOLUT_REVOGB21`)?
- `transaction_total_days` (zwykle 90) — czy starcza?
- Czy sandbox flow działa end-to-end?
- Decyzja: **adopt** (Faza 5 — auto-sync) lub **drop** (zostaje CSV-only).